# Decorators, Metaprogramming & Descriptors (5+ Years Interview Guide)
Exhaustive revision guide to closures, @functools.wraps, parameterized decorator factories, __call__ class decorators, descriptors (__get__/__set__), and __init_subclass__ on transaction classes.

### Key 5-Year Interview Concepts Covered:
- **Function Decorators**: Dedicated cell for `@functools.wraps` preserving function signatures and docstrings.
- **Parameterized & Class Decorators**: Dedicated cell for wrapper factories and `__call__()`.
- **Custom Descriptors**: Dedicated cell for `__get__`, `__set__`, and `__delete__` implementing validation protocols.
- **Metaprogramming Hooks**: Dedicated cell for `__init_subclass__` and `__new__`.

This interactive revision guide uses `data/raw_transactions.csv` with individual dedicated cells per method.

In [1]:
# Setup imports & dataset loading from raw_transactions.csv
import csv
import sys
import time
import os
import functools
import contextlib
import asyncio
import threading
from dataclasses import dataclass
from typing import List, Dict, Optional, Union, Protocol

csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
transactions = []
with open(csv_path, mode='r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        transactions.append(row)

print(f"Python Version: {sys.version.split()[0]}")
print(f"Loaded {len(transactions)} transaction records from {csv_path}")

Python Version: 3.12.7
Loaded 15000 transaction records from data/raw_transactions.csv


### Function Decorators with `@functools.wraps`
**Explanation**: Wraps functions while preserving `__name__`, `__doc__`, and parameter annotations.

**Syntax**: `@functools.wraps(func)`

In [2]:
def audit_log(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        t0 = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - t0
        print(f'[AUDIT] {func.__name__} executed in {elapsed*1000:.3f} ms')
        return result
    return wrapper

@audit_log
def process_transaction(tx_id, amount):
    """Processes and settles financial record."""
    return f'Settled {tx_id} for ${amount:,.2f}'

print(process_transaction(transactions[0]['transaction_id'], float(transactions[0]['transaction_amount'])))
print('Preserved Docstring:', process_transaction.__doc__)

[AUDIT] process_transaction executed in 0.005 ms
Settled TX110686 for $1,216.33
Preserved Docstring: Processes and settles financial record.


### Parameterized Decorator Factories & Class Decorators
**Explanation**: A 3-level closure taking decorator configuration parameters, and class-based decorators implementing `__call__`.

**Syntax**: `@retry(max_retries=3, delay=1.0)`

In [3]:
def require_min_amount(min_val):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(tx_record, *args, **kwargs):
            amt = float(tx_record['transaction_amount'])
            if amt < min_val:
                return f'SKIPPED: {tx_record["transaction_id"]} below threshold ${min_val}'
            return func(tx_record, *args, **kwargs)
        return wrapper
    return decorator

@require_min_amount(500.0)
def process_high_value(tx):
    return f'PROCESSED VIP {tx["transaction_id"]} (${float(tx["transaction_amount"]):.2f})'

print(process_high_value(transactions[0]))

PROCESSED VIP TX110686 ($1216.33)


### Custom Descriptors (`__get__`, `__set__`, `__delete__`)
**Explanation**: Descriptors intercept attribute access on classes. Essential for ORMs, validation engines, and dataclasses.

**Syntax**: `class NonNegativeAmount: def __set__(self, obj, val): ...`

In [4]:
class NonNegativeAmount:
    def __set_name__(self, owner, name):
        self.name = name
    def __get__(self, instance, owner):
        if instance is None: return self
        return instance.__dict__.get(self.name, 0.0)
    def __set__(self, instance, value):
        if value < 0.0:
            raise ValueError(f'{self.name} cannot be negative!')
        instance.__dict__[self.name] = value

class FinancialAsset:
    amount = NonNegativeAmount()
    def __init__(self, amount):
        self.amount = amount

asset = FinancialAsset(1500.0)
print('Descriptor Validated Amount:', asset.amount)

Descriptor Validated Amount: 1500.0


### Subclass Registration with `__init_subclass__`
**Explanation**: PEP 487 hook customizing and registering subclasses at definition time without complex metaclasses.

**Syntax**: `def __init_subclass__(cls, **kwargs): ...`

In [5]:
class PluginRegistry:
    plugins = {}
    def __init_subclass__(cls, plugin_name=None, **kwargs):
        super().__init_subclass__(**kwargs)
        if plugin_name:
            cls.plugins[plugin_name] = cls

class VisaHandler(PluginRegistry, plugin_name='Visa'): pass
class AmexHandler(PluginRegistry, plugin_name='Amex'): pass
print('Auto-Registered Plugins via __init_subclass__:', PluginRegistry.plugins)

Auto-Registered Plugins via __init_subclass__: {'Visa': <class '__main__.VisaHandler'>, 'Amex': <class '__main__.AmexHandler'>}


## Section: Senior Fintech Interview Scenarios (5+ Years Experience)

### Q1: Method Binding Protocol: Why `self` is passed automatically
**Explanation**: Explain that functions in Python are descriptors implementing `__get__`. When accessed via an instance (`obj.func()`), `func.__get__(obj, Class)` returns a bound method object holding `(func, obj)`.

**Syntax**: `method = obj.func; method.__self__ is obj`

In [6]:
print('Functions implement descriptor __get__ to bind instance `self` at runtime.')

Functions implement descriptor __get__ to bind instance `self` at runtime.
